In [1]:
import pandas as pd
import torch
import torch.nn.functional as F # type: ignore
from transformers import AutoTokenizer, AutoModelForSequenceClassification # type: ignore


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando il dispositivo: {device}")


Usando il dispositivo: cuda


In [3]:
# Caricare il tokenizer e il modello
tokenizer = AutoTokenizer.from_pretrained("rafalposwiata/deproberta-large-depression")
model = AutoModelForSequenceClassification.from_pretrained("rafalposwiata/deproberta-large-depression",).to(device)




In [4]:
# Funzione per classificare il testo
def classify_text(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
    probs = F.softmax(logits, dim=-1).squeeze().tolist()
    label = torch.argmax(logits, dim=-1).item()
    return label, probs[0], probs[1], probs[2]

In [5]:

df = pd.read_csv(r"C:\Users\gervi\Desktop\Tesi 04-03-2025\Work\eRisk2018_clean_no_title.csv")

In [6]:
# Applicare la funzione di classificazione
df[['Predicted_Label', 'Prob_Not_Depressed', 'Prob_Moderate_Depressed', 'Prob_Severe_Depressed']] = df['Text'].apply(lambda x: pd.Series(classify_text(x)))


In [7]:
# Rinomina la colonna 'Classification' in 'Label_User'
df.rename(columns={'Classification': 'Label_User'}, inplace=True)

In [8]:
# Salvare il nuovo DataFrame
df.to_csv(r"C:\Users\gervi\Desktop\Tesi 04-03-2025\Work\output_Classification.csv", index=False)

In [9]:
print(df.head())

   Subject ID  Chunk                 Date Title         Info  \
0           3      1  2016-01-08 13:53:09   NaN  reddit post   
1           3      1  2016-01-08 14:08:48   NaN  reddit post   
2           3      1  2016-01-11 11:38:33   NaN  reddit post   
3           3      1  2016-01-11 11:39:38   NaN  reddit post   
4           3      1  2016-01-11 11:46:02   NaN  reddit post   

                                         Text  Label_User  Predicted_Label  \
0                              Perfect pic...           0              2.0   
1             OMG... Very sad to know this...           0              2.0   
2  Messy ringtone sound from someone's mobile           0              2.0   
3      No problem... I will take care of it..           0              2.0   
4                              /r/websecurity           0              2.0   

   Prob_Not_Depressed  Prob_Moderate_Depressed  Prob_Severe_Depressed  
0            0.002533                 0.038560               0.958907  
1 